In [ ]:
from google.colab import drive
drive.mount('/content/drive')

###  Install CondaColab
Google Colab does not support Conda package management natively. This installation provides a lightweight Miniconda instance, which allows to create isolated environments with strict Python versions.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

### Create a kegacy Python environment

A new Conda environment named `legacy_cv` is created and configured to use Python version 3.8.

This is done because the  Segmenter repository relies on older computer vision libraries that are incompatible with modern Python 3.10+ environments. Using Python 3.8 ensures the code runs without syntax or dependency errors.

In [ ]:
!conda create -n legacy_cv python=3.8 -y

### Install specific deep dearning dibraries
Specific legacy versions of PyTorch (1.9.0+cu111), TorchVision, MMCV-full (1.3.17), MMSegmentation (0.17.0), `timm`, and `cityscapesscripts` are installed directly into the `legacy_cv` environment.

This is done because the Segmenter architecture depends on exact versions of the OpenMMLab libraries and PyTorch. Installing these specific historical versions prevents API mismatch errors during training and evaluation.

In [ ]:
!/usr/local/envs/legacy_cv/bin/pip install torch==1.9.0+cu111 torchvision==0.10.0+cu111 torchaudio==0.9.0 -f https://download.pytorch.org/whl/torch_stable.html
!/usr/local/envs/legacy_cv/bin/pip install timm==0.4.12 einops click pyyaml cityscapesscripts

!/usr/local/envs/legacy_cv/bin/pip install mmcv-full==1.3.17 -f https://download.openmmlab.com/mmcv/dist/cu111/torch1.9.0/index.html
!/usr/local/envs/legacy_cv/bin/pip install mmsegmentation==0.17.0

### Extract Cityscapes to local storage
The dataset's image and ground-truth `.zip` archives are unpacked from Google Drive into Colab's local `/content/dataset/cityscapes` directory.

**WARNING**: Before doing that be sure to have installed both leftImg8bit_trainvaltest.zip and gtFine_trainvaltest.zip from the official cityscapes website and uploaded them on Google Drive.

In [ ]:
!mkdir -p /content/dataset/cityscapes

# Assuming you mounted Google Drive and the zips are there:
!unzip -q /content/drive/MyDrive/leftImg8bit_trainvaltest.zip -d /content/dataset/cityscapes
!unzip -q /content/drive/MyDrive/gtFine_trainvaltest.zip -d /content/dataset/cityscapes

### 6. Process Cityscapes annotations
The `CITYSCAPES_DATASET` environment variable is set, and the official `csCreateTrainIdLabelImgs` utility is executed on the extracted files.

This is done because the raw Cityscapes dataset contains over 30 granular classes, but standard semantic segmentation evaluation ignores many of them (e.g., 'ego vehicle' or 'out of roi'). This script converts the raw annotations into the standardized 19-class format required for training and IoU calculation.

In [ ]:
!pip install cityscapesscripts

# Point the script to your unzipped dataset
%env CITYSCAPES_DATASET=/content/dataset/cityscapes

# Run the official conversion tool
!csCreateTrainIdLabelImgs

### Clone the Segmenter repository
The custom and optimized to be run on Colab's T4 GPU `segmenter` repository is downloaded from GitHub and the active directory is changed to the newly created folder.

This brings the necessary architecture code, training scripts, and memory-optimized evaluation logic directly into Colab workspace so they can be executed.

In [ ]:
!git clone https://github.com/NicolasCola7/segmenter.git
%cd segmenter

### Install the Segmenter package
The command `pip install -e .` is executed inside the `legacy_cv` environment to install the downloaded segmenter code as an editable package.

This makes the internal `segm` module globally available to the Python environment. Without this step, the training and evaluation scripts would crash with 'module not found' errors when trying to import their own functions.

In [ ]:
!/usr/local/envs/legacy_cv/bin/pip install -e .

### Launch the training loop
The `segm.train` script is invoked to train a `vit_tiny_patch16_384` backbone.

This effectively fine-tunes the Vision Transformer on the Cityscapes dataset. Strict optimizations like `batch-size 4`, `num-workers 2`, and Automatic Mixed Precision (`--amp`) are required to prevent Colab's hardware limits (16GB GPU VRAM and 12GB System RAM) from throwing Out-Of-Memory exceptions.

In [ ]:
%env DATASET=/content/dataset

!/usr/local/envs/legacy_cv/bin/python -m segm.train \
  --log-dir /content/drive/MyDrive/segmenter_logs/cityscapes_tiny \
  --dataset cityscapes \
  --backbone vit_tiny_patch16_384 \
  --decoder mask_transformer \
  --batch-size 4 \
  --amp \
  --epochs 60 \
  --eval-freq 5 \
  --num-workers 2

### Compute validation metrics
The `segm.eval.miou` script is executed twice on the saved checkpoint: once using Single Scale (`--singlescale`) prediction and once using Multi-Scale (`--multiscale`) test-time augmentation.

This step quantitatively measures the model's Mean Intersection over Union (or mIoU) on the dataset.

In [ ]:
%env DATASET=/content/dataset


# single-scale evaluation:
!/usr/local/envs/legacy_cv/bin/python -m segm.eval.miou /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/checkpoint.pth cityscapes --singlescale --num-workers 2
# multi-scale evaluation:
!/usr/local/envs/legacy_cv/bin/python -m segm.eval.miou /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/checkpoint.pth cityscapes --multiscale --num-workers 2

### Generate predictions
The `segm.inference` script is run on the 'Munster' validation city for two different saved checkpoints (the Mask Decoder model and the Linear Decoder model). Outputs are saved directly into your Google Drive.


In [ ]:
%env DATASET=/content/dataset

!mkdir -p /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs/munster_predictions
!mkdir -p /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs_linear/munster_predictions

!/usr/local/envs/legacy_cv/bin/python -m segm.inference \
  --model-path /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs/checkpoint.pth \
  -i /content/dataset/cityscapes/leftImg8bit/val/munster \
  -o /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs/munster_predictions

!/usr/local/envs/legacy_cv/bin/python -m segm.inference \
  --model-path /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs_linear/checkpoint.pth \
  -i /content/dataset/cityscapes/leftImg8bit/val/munster \
  -o /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs_linear/munster_predictions


!mkdir -p /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs/frankfurt_predictions
!mkdir -p /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs_linear/frankfurt_predictions

!/usr/local/envs/legacy_cv/bin/python -m segm.inference \
  --model-path /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs/checkpoint.pth \
  -i /content/dataset/cityscapes/leftImg8bit/val/frankfurt \
  -o /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs/frankfurt_predictions

!/usr/local/envs/legacy_cv/bin/python -m segm.inference \
  --model-path /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs_linear/checkpoint.pth \
  -i /content/dataset/cityscapes/leftImg8bit/val/frankfurt \
  -o /content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs_linear/frankfurt_predictions

### Visualize and compare results
A Python loop uses `matplotlib` and `PIL` to fetch the generated images from Google Drive and render them side-by-side.
This allows to directly visually inspect the prediction quality between the Linear Decoder and the Mask Decoder.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
import glob

# Find all the generated images
output_dir1 = "/content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs_linear/frankfurt_predictions"
images1 = sorted(glob.glob(os.path.join(output_dir1, "*.jpg")) + glob.glob(os.path.join(output_dir1, "*.png")))

output_dir2 = "/content/drive/MyDrive/segmenter_logs/cityscapes_tiny/val_default_configs_60_epochs/frankfurt_predictions"
images2 = sorted(glob.glob(os.path.join(output_dir2, "*.jpg")) + glob.glob(os.path.join(output_dir2, "*.png")))

if not images1 or not images2:
    print("No images found! Check the inference logs.")
else:
    for i, (path1, path2) in enumerate(zip(images1, images2)):


        img1 = Image.open(path1)
        img2 = Image.open(path2)

        fig, axes = plt.subplots(1, 2, figsize=(24, 8))

        axes[0].imshow(img1)
        axes[0].set_title(f"60 Epochs linear: {os.path.basename(path1)}", fontsize=12)
        axes[0].axis('off')

        axes[1].imshow(img2)
        axes[1].set_title(f"60 Epochs: {os.path.basename(path2)}", fontsize=12)
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()